In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/config.json
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/merges.txt
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/LICENSE
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/README.md
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/tokenizer.json
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/vocab.json
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/tokenizer_config.json
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/model.safetensors
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/.gitattributes
/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1/generation_config.json


In [2]:
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO_OWNER = "zak-p-y"
REPO_NAME  = "bactrian-x"
BRANCH     = "work"
CLONE_DIR  = Path(f"/kaggle/working/{REPO_NAME}")




def run(cmd, cwd=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if res.stdout:
        print(res.stdout)
    if res.stderr:
        print(res.stderr)
    res.check_returncode()
    return res


def get_repo_url():
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    return f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"


def sync_repo():
    url = get_repo_url()

    if (CLONE_DIR / ".git").exists():
        run(["git", "remote", "set-url", "origin", url], cwd=CLONE_DIR)
        run(["git", "fetch", "origin", BRANCH, "--depth", "1"], cwd=CLONE_DIR)
        run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=CLONE_DIR)
        run(["git", "clean", "-fd"], cwd=CLONE_DIR)
    else:
        if CLONE_DIR.exists():
            run(["rm", "-rf", str(CLONE_DIR)])
        run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(CLONE_DIR)])

    run(["git", "log", "--oneline", "-3"], cwd=CLONE_DIR)
    return CLONE_DIR


REPO_PATH = sync_repo()

Cloning into '/kaggle/working/bactrian-x'...

eb79730 Fix pilot OOM: gradient checkpointing and smaller micro-batch



In [3]:
import os
from pathlib import Path

# REPO_PATH is created by the sync_repo cell above.
REPO_DIR = Path(str(REPO_PATH))
os.environ["HF_HOME"] = "/kaggle/working/.hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/.hf/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Skip torch.compile: on T4 it costs minutes, can hit Triton issues and does not
# speed up these short LoRA runs.
os.environ["TORCH_COMPILE"] = "0"
os.environ["WANDB_DISABLED"] = "true"
# Reduce allocator fragmentation (the ~330 MiB "reserved but unallocated" in the OOM log).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in ("/kaggle/working/out", "/kaggle/working/results", os.environ["HF_HOME"]):
    os.makedirs(d, exist_ok=True)

print("repo:", REPO_DIR)
print("HF_HOME:", os.environ["HF_HOME"])
print("TORCH_COMPILE:", os.environ["TORCH_COMPILE"])
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])


repo: /kaggle/working/bactrian-x
HF_HOME: /kaggle/working/.hf
TORCH_COMPILE: 0
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [4]:
import subprocess, sys

# Kaggle images run Python 3.13. transformers 4.44 pins tokenizers<0.20, which has
# no cp313 wheels and fails to build from source. These versions ship cp313 wheels
# and still support Qwen2.5 + peft.
PACKAGES = [
    "transformers==4.49.0",
    "peft==0.14.0",
    "accelerate==1.4.0",
    "datasets==3.3.2",
    "sentencepiece",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *PACKAGES], check=True)

check = subprocess.run(
    [sys.executable, "-c",
     "import transformers, peft, datasets, accelerate, tokenizers;"
     "print('transformers', transformers.__version__);"
     "print('peft', peft.__version__);"
     "print('datasets', datasets.__version__);"
     "print('accelerate', accelerate.__version__);"
     "print('tokenizers', tokenizers.__version__);"
     "from peft import prepare_model_for_kbit_training;"
     "print('peft import OK')"],
    capture_output=True, text=True,
)
print(check.stdout)
print(check.stderr)
check.check_returncode()


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.12.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but yo

transformers 4.49.0
peft 0.14.0
datasets 3.3.2
accelerate 1.4.0
tokenizers 0.21.4
peft import OK




In [5]:
import os, sys, subprocess, json, glob, csv, time
from pathlib import Path

print(sys.version)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), round(props.total_memory / 2**30, 1), "GiB")

print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)


3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Thu Nov  5 16:39:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A

In [6]:
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path(str(REPO_PATH))


def run_stream(cmd, cwd, env=None):
    """Run a command and stream stdout/stderr into the notebook."""
    merged_env = {**os.environ, **(env or {})}
    print("$", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        [str(c) for c in cmd], cwd=cwd, env=merged_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


SMOKE_DIR = "/kaggle/working/out/smoke"

run_stream([
    sys.executable, "finetune.py",
    "--model_name_or_path", "Qwen/Qwen2.5-1.5B",
    "--lang", "en",
    "--max_samples_per_lang", "200",
    "--output_dir", SMOKE_DIR,
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "2",
    "--num_train_epochs", "1",
    "--max_steps", "10",
    "--warmup_steps", "2",
    "--model_max_length", "512",
    "--val_set_size", "50",
    "--learning_rate", "2e-4",
    "--lora_r", "8",
    "--lora_alpha", "16",
    "--lora_target_modules", "q_proj,k_proj,v_proj,o_proj",
    "--save_steps", "10",
    "--save_total_limit", "1",
    "--logging_steps", "5",
    "--report_to", "none",
], cwd=REPO_DIR, env={"CUDA_VISIBLE_DEVICES": "0"})

print(sorted(os.listdir(SMOKE_DIR)))


$ /usr/bin/python3 finetune.py --model_name_or_path Qwen/Qwen2.5-1.5B --lang en --max_samples_per_lang 200 --output_dir /kaggle/working/out/smoke --per_device_train_batch_size 2 --gradient_accumulation_steps 2 --num_train_epochs 1 --max_steps 10 --warmup_steps 2 --model_max_length 512 --val_set_size 50 --learning_rate 2e-4 --lora_r 8 --lora_alpha 16 --lora_target_modules q_proj,k_proj,v_proj,o_proj --save_steps 10 --save_total_limit 1 --logging_steps 5 --report_to none
INFO:__main__:Model parameters ModelArguments(model_name_or_path='Qwen/Qwen2.5-1.5B', load_in_8bit=False, lora_r=8, lora_alpha=16, lora_dropout=0.05, lora_target_modules='q_proj,k_proj,v_proj,o_proj')
INFO:__main__:Training/evaluation parameters BactrianTrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=

In [7]:
import os, sys
from pathlib import Path

LANGS = "en,ru"
N_PER_LANG = 25_000      # out of 67k available per language
EPOCHS = 2
LR = 2e-4
SEEDS = [42, 43]
RANKS = [8, 64]
OUT_ROOT = Path("/kaggle/working/out")

# Memory: batch 4 x seq 1024 OOMs on T4 (activations ~10 GiB + logits 1.2 GiB fp16
# + fp32 copy + 3.1 GiB weights > 14.56 GiB). Gradient checkpointing cuts activations
# to ~1 GiB; batch 2 keeps a safety margin. Effective batch stays 128.
GC_KWARGS = '{"use_reentrant": false}'

for rank in RANKS:
    alpha = 2 * rank          # keep alpha / r constant
    for seed in SEEDS:
        out_dir = OUT_ROOT / f"qwen1.5b-r{rank}-s{seed}"
        run_stream([
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node=2", "--master_port=1234", "finetune.py",
            "--model_name_or_path", "Qwen/Qwen2.5-1.5B",
            "--lang", LANGS,
            "--max_samples_per_lang", str(N_PER_LANG),
            "--output_dir", str(out_dir),
            "--per_device_train_batch_size", "2",
            "--gradient_accumulation_steps", "32",
            "--gradient_checkpointing", "True",
            "--gradient_checkpointing_kwargs", GC_KWARGS,
            "--per_device_eval_batch_size", "2",
            "--num_train_epochs", str(EPOCHS),
            "--model_max_length", "1024",
            "--learning_rate", str(LR),
            "--val_set_size", "1000",
            "--warmup_steps", "50",
            "--lora_r", str(rank),
            "--lora_alpha", str(alpha),
            "--lora_dropout", "0.05",
            "--lora_target_modules", "q_proj,k_proj,v_proj,o_proj",
            "--save_steps", "200",
            "--eval_steps", "200",
            "--save_total_limit", "2",
            "--group_by_length",
            "--fp16",
            "--report_to", "none",
            "--seed", str(seed),
        ], cwd=REPO_DIR)


$ /usr/bin/python3 -m torch.distributed.run --nproc_per_node=2 --master_port=1234 finetune.py --model_name_or_path Qwen/Qwen2.5-1.5B --lang en,ru --max_samples_per_lang 25000 --output_dir /kaggle/working/out/qwen1.5b-r8-s42 --per_device_train_batch_size 2 --gradient_accumulation_steps 32 --gradient_checkpointing True --gradient_checkpointing_kwargs {"use_reentrant": false} --per_device_eval_batch_size 2 --num_train_epochs 2 --model_max_length 1024 --learning_rate 0.0002 --val_set_size 1000 --warmup_steps 50 --lora_r 8 --lora_alpha 16 --lora_dropout 0.05 --lora_target_modules q_proj,k_proj,v_proj,o_proj --save_steps 200 --eval_steps 200 --save_total_limit 2 --group_by_length --fp16 --report_to none --seed 42

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
******************************

In [8]:
import glob, json, os, csv

rows = []
for out_dir in sorted(glob.glob("/kaggle/working/out/qwen1.5b-r*")):
    states = sorted(glob.glob(os.path.join(out_dir, "checkpoint-*", "trainer_state.json")))
    cfg_path = os.path.join(out_dir, "adapter_config.json")
    if not states or not os.path.exists(cfg_path):
        print("skipping incomplete run:", out_dir)
        continue
    state = json.load(open(states[-1]))
    cfg = json.load(open(cfg_path))
    history = state.get("log_history", [])
    eval_losses = [h["eval_loss"] for h in history if "eval_loss" in h]
    train_losses = [h["train_loss"] for h in history if "train_loss" in h]
    rows.append({
        "run": os.path.basename(out_dir),
        "r": cfg["r"],
        "alpha": cfg["lora_alpha"],
        "target_modules": ",".join(cfg["target_modules"]),
        "global_step": state.get("global_step"),
        "epoch": state.get("epoch"),
        "last_train_loss": train_losses[-1] if train_losses else None,
        "final_eval_loss": eval_losses[-1] if eval_losses else None,
        "min_eval_loss": min(eval_losses) if eval_losses else None,
    })

csv_path = "/kaggle/working/results/pilot.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["run"])
    writer.writeheader()
    writer.writerows(rows)

print(open(csv_path).read())


run,r,alpha,target_modules,global_step,epoch,last_train_loss,final_eval_loss,min_eval_loss
qwen1.5b-r64-s42,64,128,"k_proj,v_proj,o_proj,q_proj",764,1.9978775510204083,,1.0879945755004883,1.0879945755004883
qwen1.5b-r64-s43,64,128,"k_proj,q_proj,v_proj,o_proj",764,1.9978775510204083,,1.0904697179794312,1.0904697179794312
qwen1.5b-r8-s42,8,16,"q_proj,k_proj,o_proj,v_proj",764,1.9978775510204083,,1.0957427024841309,1.0957427024841309
qwen1.5b-r8-s43,8,16,"q_proj,v_proj,k_proj,o_proj",764,1.9978775510204083,,1.0977731943130493,1.0977731943130493

